# IOS Risk — Project 03 Evaluation

Scores the fine-tuned adapter against Project 03's success criteria and against
the Project 01 XGBoost baseline.

**Targets:** tier accuracy > 0.70 · avg quality > 0.60
**Baseline:** XGBoost — P 0.9011 · R 0.8367 · F1 0.8677

Runs BOTH the base model and the tuned adapter over the same 250 held-out cases,
so the comparison is like-for-like. ~20 minutes on a T4.

**Before running:** Settings -> Accelerator -> GPU T4 x2, and attach the training
run's output via Add Input -> Notebooks.

### Step 1: Install (same pins as training)

In [ ]:
!pip install \
    "transformers==5.5.0" \
    "datasets==4.3.0" \
    "trl==0.24.0" \
    "bitsandbytes==0.50.1" \
    "xformers==0.0.34" \
    "peft>=0.18.0" \
    unsloth unsloth_zoo 2>&1 | tail -12

### Step 2: Pull the eval code from GitHub\nKeeps the notebook thin and the scoring logic version-controlled.

In [ ]:
!rm -rf /kaggle/working/core
!git clone -q https://github.com/Etherlabs-dev/domain-intelligence-core.git /kaggle/working/core
import sys, os
sys.path.insert(0, "/kaggle/working/core")
os.chdir("/kaggle/working/core")
print("eval/ contents:", os.listdir("eval"))

### Step 3: Locate the trained adapter\nSame auto-discovery as training — the mount path is not stable across Kaggle UI versions.

In [ ]:
import os

def find_adapter(name_contains="ios-risk-llama3-v2"):
    hits = []
    for root, _dirs, files in os.walk("/kaggle/input"):
        if "adapter_config.json" in files:
            hits.append(root)
    exact = [h for h in hits if name_contains in h and "checkpoint" not in h]
    if exact:
        return exact[0]
    if hits:
        print("adapters found:", hits)
        return hits[0]
    return None

ADAPTER = find_adapter()
if ADAPTER is None:
    for root, _d, _f in os.walk("/kaggle/input"):
        if root.rstrip("/").count("/") - 2 <= 4:
            print("   ", root)
    raise AssertionError(
        "No adapter under /kaggle/input. "
        "Attach it: Add Input -> Notebooks -> the training run version."
    )
print("adapter:", ADAPTER)

### Step 4: Score the BASE model\nThe control. Whatever the fine-tune scores only means something relative to this.

In [ ]:
import unsloth  # must import before transformers/trl
from eval.domain_eval import run

base_summary = run(
    model_id="unsloth/Meta-Llama-3.1-8B-Instruct",
    tag="base",
    testset_path="eval/testset.json",
    out_dir="/kaggle/working/eval_results",
)

### Step 5: Score the TUNED adapter

In [ ]:
tuned_summary = run(
    model_id=ADAPTER,
    tag="tuned",
    testset_path="eval/testset.json",
    out_dir="/kaggle/working/eval_results",
)

### Step 6: Verdict

In [ ]:
import json

def row(label, b, t, target=None):
    flag = ""
    if target is not None:
        flag = "  PASS" if t > target else "  FAIL"
    print(f"{label:<18} base {b:>7.4f}   tuned {t:>7.4f}   delta {t-b:>+7.4f}{flag}")

ra_b, ra_t = base_summary["risk_assessment"], tuned_summary["risk_assessment"]
print("RISK ASSESSMENT  (50 held-out scenario prompts)")
row("tier accuracy", ra_b["tier_accuracy"], ra_t["tier_accuracy"], 0.70)
row("avg quality",   ra_b["avg_quality"],   ra_t["avg_quality"],   0.60)
row("reasoning rate", ra_b["reasoning_rate"], ra_t["reasoning_rate"])
row("action rate",    ra_b["action_rate"],    ra_t["action_rate"])

cb, ct = base_summary["classification"], tuned_summary["classification"]
print("\nCLASSIFICATION  (200 balanced held-out transactions)")
row("precision", cb["precision"], ct["precision"])
row("recall",    cb["recall"],    ct["recall"])
row("f1",        cb["f1"],        ct["f1"])
print(f"{'':<18} XGBoost (Project 01):  P 0.9011  R 0.8367  F1 0.8677")
print(f"unparseable responses — base {cb['unparseable_responses']}, tuned {ct['unparseable_responses']}")

print("\nPROJECT 03 CRITERIA:", "PASS" if tuned_summary["passes_project03"] else "FAIL")

with open("/kaggle/working/eval_results/comparison.json", "w") as f:
    json.dump({"base": base_summary, "tuned": tuned_summary}, f, indent=2)
print("\nwritten -> /kaggle/working/eval_results/")

### Step 7: Read actual outputs

Metrics hide behaviour. The v2 smoke test looked correct but invented
"Probability of fraud: 89.4%" and "Total score: 68" — numbers with no basis in
the training data or the input. Scan for that here.

In [ ]:
import json
rows = json.load(open("/kaggle/working/eval_results/eval_tuned.json"))["risk_rows"]

for r in rows[:5]:
    ok = "OK  " if r["correct_tier"] else "MISS"
    print(f"[{ok}] expected {r['expected_tier']:<8} got {str(r['predicted_tier']):<8} ({r['expected_pattern']})")
    print("   ", r["input"])
    print("   ", r["response"][:300].replace("\n", " "))
    print()

import re
suspect = [r for r in rows if re.search(r"\d+(\.\d+)?%|score:\s*\d+", r["response"], re.I)]
print(f"responses containing an invented probability or score: {len(suspect)}/{len(rows)}")